---

Reawakening

---

In [ ]:
# autoload
%load_ext autoreload
%autoreload 2

# import pgl commands
from pgl import pgl, pglExperiment, pglEyeTrackingCalibrationTask, pglMessageAckTask, pglImageDatabase, pglMessages, pglTask, pglLabJack, pglParameter

# import other libraries
import numpy as np

# initialize pgl
pgl = pgl()
pgl.cleanUp()

---

Display settings

---

In [ ]:
# display settings - LCD should show up as SAMSUNG (select from top)
pgl.displaySettings()

---

Settings

---

In [ ]:
# Experiment settings, select reawakening
pgl.settings()

---

Eye tracker settings

---

In [ ]:
# set eye tracker settings, for example how many calibration points and the screen area to use for calibration
pgl.eyeTrackerSettings(settingsName='reawakening')

---

Search task

---

In [ ]:
class pglSearchTask(pglTask):
    
    ########################
    def __init__(self, pgl):
        super().__init__(pgl)
        
        # set task parameters, these will automatically be saved in the settings file
        self.settings.taskName = "Search Task"
        self.settings.nTrials = 100
        
        # fixed parameters, these will automatically be saved in the settings file
        self.settings.config.imagesDirectory = "/Users/justin/Desktop/reawakening/stimulus/chair/0001"
        self.settings.config.imageHeight= 15
        
        # setup digitalIO
        #from pgl import pglLabJack
        #self.digIO = pglLabJack()
        
        # set seglens, 
        # 1st segment is fixation
        # 2nd segment is image display
        # 2nd segment is ISI
        self.settings.seglen = [0.5, 3.0, 0.5]

        # initialize image database using parameters from config
        self.state._imdb = pglImageDatabase(self.settings.config.imagesDirectory)
        if self.state._imdb.nStimuli==0:
            pglMessages.warning(f"No images found in {self.state._imdb.dataPath}")
            return
                
        # preload images
        self.state._imdb.preload()            

        # add parameter for image number
        imageNumParameter = pglParameter('imageNum',np.arange(self.state._imdb.nStimuli))
        self.addParameter(imageNumParameter)
        
        imageNumParameter.print()
    ########################
    def startSegment(self, startTime):
        '''
        Start a segment
        '''
        super().startSegment(startTime)
    
        # load the image
        if self.state.currentSegment == 0: 
            # set fixation color to white
            self.state.fixColor = 1
            # get the current image number
            imageNum = self.currentParams['imageNum']
            # get the image data
            img = self.state._imdb.get(imageNum)
            img.convert("RGB")
            print(f"img: {img}")
            # turn into a pglImage
            self.state._currentImage = self.pgl.imageCreate(np.array(img))

    ########################
    # handleSubjectResponse
    ########################    
    def handleSubjectResponse(self, response, updateTime):
        '''
        Handle the subject response. Response will come in as an integer
        value of what button was pressed. The order of buttons is set
        in pgl.settings() in the field "responseKeys"
        '''
        # already received a response
        if self.state.gotResponse: return None
        # mark that we got a response
        self.state.gotResponse = True
        
        # return response type
        return True

    ########################
    # updateScren
    ########################
    def updateScreen(self):
        '''
        update the screen
        '''
        if self.state.currentSegment == 1: 
            if self.state._currentImage:
                self.state._currentImage.display(height=self.settings.config.imageHeight)
        else:
            pgl.fixationABC()


---

Setup experiment

---

In [ ]:
# clean up any open windows
pgl.cleanUp()

#e = pglExperiment(pgl,settingsName='Cinema',experimentName='imageTask')
e = pglExperiment(pgl,experimentName='Search Task',settingsName='reawakening')

# First run a calibration
messageAckTask = pglMessageAckTask(pgl, "Press space to start eye calibration")
e.addTask(messageAckTask,addPhase=True)
calibrationTask = pglEyeTrackingCalibrationTask(pgl)
e.addTask(calibrationTask,addPhase=True)

# add the search task to the experiment
#messageAckTask = pglMessageAckTask(pgl, "Press space to start search task")
#e.addTask(messageAckTask,addPhase=True)
#searchTask = pglSearchTask(pgl)
#e.addTask(searchTask,addPhase=True)

# Run a final calibration
messageAckTask = pglMessageAckTask(pgl, "Press space to start eye calibration")
e.addTask(messageAckTask,addPhase=True)
calibrationTask = pglEyeTrackingCalibrationTask(pgl)
e.addTask(calibrationTask,addPhase=True)

# Run a final calibration
messageAckTask = pglMessageAckTask(pgl, "Press space to start eye calibration")
e.addTask(messageAckTask,addPhase=True)
calibrationTask = pglEyeTrackingCalibrationTask(pgl)
e.addTask(calibrationTask,addPhase=True)

# Run a final calibration
messageAckTask = pglMessageAckTask(pgl, "Press space to start eye calibration")
e.addTask(messageAckTask,addPhase=True)
calibrationTask = pglEyeTrackingCalibrationTask(pgl)
e.addTask(calibrationTask,addPhase=True)

# Run a final calibration
messageAckTask = pglMessageAckTask(pgl, "Press space to start eye calibration")
e.addTask(messageAckTask,addPhase=True)
calibrationTask = pglEyeTrackingCalibrationTask(pgl)
e.addTask(calibrationTask,addPhase=True)


---

Run task

---

In [ ]:
e.initScreen()
e.run()
e.display()


---

Test Labjack

---

In [ ]:
from pgl import pgl, pglLabJack
pgl = pgl()
labJack = pglLabJack()

In [ ]:
labJack.setupDigitalOutput(channel=0, group="FI0", pulseLen=3)

In [ ]:
labJack.digitalOutputPulse(0)